# 과제 2. 송파구 택시·대중교통 통행 비교

송파구 안에서 출발하고 도착하는 30명의 통행을 비교합니다. 출발시각은 모두 오전 9시입니다.
각 사람이 택시를 타면 얼마나 걸리는지, 대중교통을 타면 얼마나 걸리는지 구합니다.
이동거리도 비교하고, 두 경우의 이동 모습을 지도에서 나란히 재생합니다.

3~7장에서 배운 내용을 바탕으로 하는 개인 과제입니다.
다익스트라와 RAPTOR는 실습 함수를 사용합니다. 데이터와 지도 재생 코드는 과제와 함께 제공합니다.
예상 소요 시간은 2~3시간입니다.

> 이 노트북의 제공 코드를 실행하고 각 문제 아래에 답안을 작성합니다.
> [제공 데이터와 출처](https://github.com/jihoyeo/mobility-simulation-book/blob/main/docs/ASSIGNMENT2_DATA.md)를 읽고 계산 조건을 확인합니다.
> O-D 30건은 가상 통행이며, 실제 승객의 이동 기록이 아닙니다.

학번:  
이름:

웹에서 자료를 받으려면 [O-D CSV](https://raw.githubusercontent.com/jihoyeo/mobility-simulation-book/main/assignment/data/od.csv)와 [도로망·시간표 ZIP](https://raw.githubusercontent.com/jihoyeo/mobility-simulation-book/main/assignment/data/songpa_network.zip)을 사용합니다.
노트북은 이 저장소의 `smartmob` 패키지와 함께 실행합니다.


## 데이터와 계산 조건

송파구 OSM 도로망, 대중교통 시간표인 GTFS, 출발지·목적지(O-D) 30쌍을 사용합니다.
경로를 계산할 수 있도록 각 출발지와 목적지를 가장 가까운 도로망 노드로 옮겨 두었습니다. 이를 스냅이라고 합니다.
택시와 대중교통 모두 옮긴 좌표를 출발지와 목적지로 사용합니다. 원래 좌표에서 옮긴 좌표까지 이동하는 시간은 제외합니다.

한 사람에게는 출발지와 목적지가 하나씩 있습니다. 한 사람이 여러 곳을 차례로 방문하는 경로를 구하는 과제는 아닙니다.
출발지와 목적지는 송파구 안에 있지만, 중간에 구 밖을 지날 수 있어 도로망과 GTFS에는 주변 지역도 포함합니다.
도로망은 서울 캐시에서 송파구 주변 4km까지 추출했으며, 경기도 쪽 도로는 완전하지 않습니다.
출발지·목적지는 제공된 2013년 송파구 경계 안에서 생성했습니다.

계산 조건은 다음과 같습니다.

| 항목 | 적용 값 | 계산 범위 |
|---|---|---|
| 출발시각 | 오전 9시 | 2024년 10월 7일 운행, 09:00 출발 |
| 택시 속도 | 30km/h | 모든 도로에서 고정, 실제 혼잡속도는 미반영 |
| 택시 대기 | 0분 | 출발지에서 즉시 탑승, 배차·주차시간 제외 |
| 보행속도 | 1.2m/s | 접근·환승·도착 도보에 공통 적용 |
| 도보거리 | 직선거리의 1.35배 | 실제 보행로 연결 여부는 미반영 |
| 접근·도착 도보 상한 | 각각 800m | 출발지에서 첫 정류장, 마지막 정류장에서 목적지 |
| 도보 환승 상한 | 500m | 정류장 사이 이동 |
| 환승 상한 | 2회 | RAPTOR에서 최대 3회 탑승 |

택시 이동거리는 경로에 포함된 도로 엣지의 길이를 더해 구합니다.
버스·열차로 이동한 거리는 경유한 정류장을 직선으로 연결해 계산합니다. 여기에 걷는 거리를 더하면 대중교통 총 이동거리입니다.
도로를 따라 계산한 택시 거리와 달리, 대중교통 거리는 실제 노선이 돌아가는 구간을 반영하지 못합니다.


## 실행 준비

이 노트북에서 과제 안내를 읽고 제공 코드를 실행한 뒤 답안을 작성합니다.
데이터는 이 노트북 옆의 `data/`에 두 파일로 들어 있습니다. `od.csv`는 30명의 통행 목록이고, `songpa_network.zip`은 도로망과 시간표입니다. ZIP은 풀지 않아도 됩니다. 별도 서버나 DTUMOS 설치는 필요하지 않습니다.
저장소의 `requirements.txt`에 있는 패키지를 설치한 환경에서 실행합니다.

다익스트라와 RAPTOR는 기존 함수를 import합니다. 한 사람의 경로 계산과 지도 재생까지 제공하며,
30명 반복 계산, 비교표·그래프 작성, 결과 해석은 각 문제 아래의 학생 작업 셀에서 진행합니다.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from smartmob.assignment2 import load_data, load_boundary, taxi_route, transit_route, export_playback

DATA = ROOT / "assignment" / "data"
OUTPUT = ROOT / "outputs" / "assignment2_student"
OUTPUT.mkdir(parents=True, exist_ok=True)

### 데이터 불러오기

2024년 GTFS에서 2024년 10월 7일의 운행을 골랐습니다. 출발시각은 오전 9시입니다.
O-D 30건은 난수 시드 42로 만든 가상 통행이며, 두 수단 모두 경로가 있는 경우만 남겼습니다.
따라서 송파구 주민의 실제 통행 분포를 나타내지 않습니다.

도로망은 서울 캐시 중 송파구와 주변 4km에 걸치는 도로입니다. 경기도 쪽 도로는 완전하지 않습니다.
송파구 안팎 판정에는 2013년 공개 경계를 사용했습니다. 최신 행정경계로 해석하지 않습니다.
GTFS 노선은 경계에서 자르지 않고 전체 정류장 순서를 보존했습니다.
[데이터 설명서](https://github.com/jihoyeo/mobility-simulation-book/blob/main/docs/ASSIGNMENT2_DATA.md)에 출처와 나머지 계산 조건이 있습니다.

시간표를 RAPTOR 자료구조로 바꾸는 첫 실행에는 수십 초 정도 걸릴 수 있습니다.

In [ ]:
graph, transit, od = load_data(DATA)
display(od.head())
print("인원:", od.person_id.nunique())
print("도로 노드:", len(graph.coord), "도로 엣지:", len(graph.edges))
print(transit.describe())

출발지와 목적지의 원래 좌표는 `origin_raw_lat/lon`, `destination_raw_lat/lon`입니다.
경로 계산에는 스냅한 `origin_lat/lon`, `destination_lat/lon`을 사용합니다.
`origin_snap_m`, `destination_snap_m`은 옮긴 거리입니다.

다음은 출발지와 목적지를 그리는 기본 지도입니다. 도로와 경계는 로컬 파일에서 읽습니다.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

boundary = gpd.GeoDataFrame.from_features(load_boundary(DATA)["features"], crs="EPSG:4326")
fig, ax = plt.subplots(figsize=(8, 8))
boundary.boundary.plot(ax=ax, color="gray", linewidth=1)
ax.scatter(od.origin_lon, od.origin_lat, label="Origin", s=25)
ax.scatter(od.destination_lon, od.destination_lat, label="Destination", marker="x", s=30)
ax.set(xlabel="Longitude", ylabel="Latitude", title="Assignment 2: synthetic O-D")
ax.legend()
fig.savefig(OUTPUT / "od_map.png", dpi=150, bbox_inches="tight")
plt.show()

## 문제 1. 통행시간과 거리 비교 (60점)

### 1-1. 입력 자료 확인 (10점)

O-D 표의 인원과 출발시각을 확인하고, 스냅한 출발지와 목적지를 지도에 표시합니다.
다음 항목을 적습니다.

- O-D 행 수와 서로 다른 사람 식별자의 수
- 원래 좌표에서 스냅 좌표까지의 최대 거리
- GTFS 기준 운행일과 택시 속도 가정

지도 1장과 확인한 내용을 4~5줄로 정리합니다.

### 1-2. 경로 계산 (25점)

택시는 다익스트라로 가장 빨리 도착하는 도로 경로를 구합니다. 대중교통은 RAPTOR를 사용합니다.
먼저 한 사람의 경로를 구한 뒤, 같은 방법으로 30명의 경로를 계산합니다.

대중교통은 정류장에 도착한 뒤에도 목적지까지 걸어야 합니다.
도착 정류장 후보마다 마지막 도보시간을 더하고, 목적지에 가장 일찍 도착하는 경로를 고릅니다.
목적지 도착시각에서 오전 9시를 빼면 총 통행시간입니다. 이 값이 도보·대기·차내시간의 합과 같은지 확인합니다.

결과는 사람당 한 행으로 정리합니다. 시간은 분, 거리는 km로 표시합니다.
표에 넣을 항목은 다음과 같습니다.

- 사람 식별자와 수단별 경로 탐색 성공 여부
- 택시 통행시간과 이동거리
- 대중교통 총 통행시간, 도보시간, 대기시간, 차내시간, 환승 횟수
- 대중교통 총 이동거리, 차내 거리, 도보거리
- 시간 차이와 거리 차이: 대중교통 값에서 택시 값을 뺀 값

경로를 찾지 못한 경우에는 결측값과 실패 이유를 남깁니다.
평균을 구할 때는 택시와 대중교통 경로를 모두 찾은 사람만 포함합니다. 몇 명의 평균인지도 적습니다.

### 1-3. 결과 해석 (25점)

두 수단의 평균 통행시간과 평균 이동거리를 구합니다.
사람별 시간 차이와 거리 차이를 각각 막대그래프로 그립니다.
시간 차이가 가장 큰 사람 한 명을 골라, 아래 질문에 대한 답을 5~8문장으로 적습니다.

1. 두 수단의 통행시간과 거리는 각각 얼마입니까?
2. 대중교통의 도보·대기·차내시간은 각각 얼마입니까?
3. 거리 차이만으로 시간 차이를 설명할 수 있습니까?
4. 택시를 기다리는 시간이 생기거나 요금까지 비교한다면, 어느 수단이 낫다는 판단이 달라질까요? 그 이유는 무엇입니까?


### 제공 예제: 한 사람의 경로

`taxi_route`는 다익스트라를 호출하고, 선택한 도로 엣지의 길이와 통행시간을 합합니다.
속도는 모든 엣지에서 30km/h입니다. 도로 형상도 함께 반환합니다.

`transit_route`는 RAPTOR를 호출합니다. 마지막 정류장에서 목적지까지 걷는 시간을 더해
가장 일찍 목적지에 도착하는 경로를 고릅니다. 접근·도착 도보는 각각 800m, 환승 도보는 500m,
환승은 2회까지 허용합니다. 걷는 거리는 직선거리의 1.35배이고 보행속도는 1.2m/s입니다.

내부에서는 `smartmob.teaching.dijkstra.dijkstra`와 `smartmob.teaching.raptor.raptor`를 사용합니다.
과제에서는 아래 호출 방법을 그대로 사용해도 됩니다.

In [ ]:
person = od.iloc[0]
taxi = taxi_route(graph, person.origin_node, person.destination_node)
public = transit_route(
    transit,
    (person.origin_lat, person.origin_lon),
    (person.destination_lat, person.destination_lon),
)
example = {"person_id": person.person_id, "taxi": taxi, "transit": public}
display(pd.DataFrame([
    {"mode": "taxi", "minutes": taxi["duration_min"], "km": taxi["distance_km"]},
    {"mode": "transit", "minutes": public["duration_min"], "km": public["distance_km"]},
]))

`duration_min`은 총 통행시간(분), `distance_km`는 이동거리(km)입니다.
대중교통 결과에는 `walk_min`, `wait_min`, `in_vehicle_min`, `transfers`,
`walk_km`, `in_vehicle_km`, `routes`도 들어 있습니다.
`segments`는 지도 재생에 쓰는 구간별 시각·좌표이므로 그대로 보관합니다.

도보·대기·차내시간을 더하면 총 통행시간이 되는지 확인합니다.

In [ ]:
parts = {k: public[k] for k in ("walk_min", "wait_min", "in_vehicle_min", "transfers", "routes")}
print(parts)
total = sum(public[k] for k in ("walk_min", "wait_min", "in_vehicle_min"))
print("시간 합:", total, "총 통행시간:", public["duration_min"])
assert abs(total - public["duration_min"]) < 1e-6

### 답안 작성: 30명 비교

위의 한 사람 계산을 반복합니다. 결과 목록에는 각 사람의 `person_id`, `taxi`, `transit` 결과를 넣습니다.
경로를 찾지 못한 경우에는 실패 이유를 기록하고, 지표에는 결측값을 사용합니다.
`smartmob.assignment2.compare_person`을 사용하면 실패 기록까지 같은 형식으로 받을 수 있습니다.

1. `od`의 각 행에 대해 두 수단의 경로를 계산해 `results`에 모읍니다.
2. 결과에서 지표를 꺼내 사람당 한 행인 비교표를 만듭니다.
3. 대중교통 값에서 택시 값을 빼 시간·거리 차이를 계산합니다.
4. 두 수단 모두 성공한 사람들의 평균을 구하고 사람별 차이를 그래프로 그립니다.

아래 셀에 작성합니다. 비교표를 `comparison.csv`로 저장하고, 해석은 별도 Markdown 셀에 적습니다.

In [ ]:
from smartmob.assignment2 import compare_person

results = []
# 여기에 30명 반복 계산과 비교표 작성을 추가합니다.

### 해석과 답안

이곳에 계산 결과와 해석을 작성합니다.

## 문제 2. 30명의 이동 시각화 (40점)

### 2-1. 이동 재생 (25점)

문제 1에서 구한 경로와 이동시각을 제공된 지도 재생 코드에 넣습니다.
왼쪽 지도에는 30명이 택시를 탄 경우를, 오른쪽에는 대중교통을 이용한 경우를 표시합니다.
두 지도는 같은 범위를 보여 주고, 같은 시각을 재생해야 합니다. 오전 9시에 시작해 마지막 사람이 도착할 때까지 재생합니다.

지도 위의 점 하나가 사람 한 명입니다. 두 지도에서 같은 사람을 찾을 수 있도록 같은 식별자를 붙입니다.
택시를 탄 사람의 점은 계산한 도로 경로를 구간별 통행시간에 맞춰 움직입니다.
대중교통을 이용하는 사람은 걷는 중, 기다리는 중, 차에 탄 상태, 도착한 상태를 구별해 표시합니다.
기다리는 동안에는 정류장에 머물고, 도착한 뒤에는 목적지에 남아 있습니다.
화면에는 현재 시각과 수단별 도착 인원수를 표시합니다.

버스·열차를 탄 동안에는 경유 정류장을 차례로 잇는 선 위에서 점이 움직입니다.
시간표의 출발·도착시각에 맞춰 정류장 사이 위치를 보간하고, 정차 중에는 점을 멈춥니다.
이 선은 실제 GPS 이동 기록이 아닙니다. 같은 차량에 탄 사람들의 점은 겹칠 수 있습니다.

두 지도를 함께 재생하는 HTML 파일 1개 또는 동영상 1개를 제출합니다.
재생 기능을 직접 구현할 필요는 없습니다.

### 2-2. 장면 비교 (15점)

오전 9시, 9시 10분, 9시 20분의 화면을 저장합니다.
해당 시각 전에 모두 도착했어도 화면을 저장합니다. 도착한 사람의 점은 목적지에 남겨 둡니다.
각 화면에는 두 지도와 수단별 도착 인원수가 보여야 합니다.

문제 1에서 고른 사람이 세 시각에 각각 어디에 있는지, 걷거나 기다리거나 차를 타고 있는지 비교합니다.
정류장에서 기다리는 동안에도 점이 움직인다면, 화면이 계산 결과를 어떻게 잘못 보여 주는지 설명합니다.
화면 3장과 설명 3~5문장을 노트북에 넣습니다.


### 제공 예제: 한 사람의 이동 재생

저장된 HTML을 브라우저에서 엽니다. 외부 지도 타일이나 인터넷 연결이 필요하지 않습니다.
두 지도에서 같은 번호의 점이 같은 사람입니다. 09:00·09:10·09:20 버튼으로 시각을 바꾸고,
사람 번호를 선택하면 두 수단의 경로와 현재 상태를 볼 수 있습니다.
`화면 PNG 저장`으로 현재 장면을 저장합니다.

In [ ]:
preview = export_playback(
    graph, od.iloc[:1], [example], OUTPUT / "one_person.html"
)
print("재생 파일:", preview.relative_to(ROOT).as_posix())

### 답안 작성: 30명 지도 재생

`results`를 완성하면 아래 셀에서 재생 파일을 만듭니다.
지도에서 저장한 장면 세 장과 과제의 질문에 대한 답을 제출 노트북에 넣습니다.

In [ ]:
if len(results) == 30:
    playback = export_playback(graph, od, results, OUTPUT / "playback.html")
    print("저장:", playback)
else:
    print("30명 비교 셀에서 results를 먼저 작성합니다.")

### 해석과 답안

이곳에 계산 결과와 해석을 작성합니다.

## 제출물과 채점 기준

다음 세 파일을 제출합니다.

- 실행 결과와 답안을 작성한 노트북: 지도, 그래프, 재생 화면 3장 포함
- 사람별 비교 결과 CSV: 30행
- 이동 시각화 HTML 또는 동영상

| 항목 | 채점 내용 | 배점 |
|---|---|---:|
| 입력 자료 | 인원·출발시각·스냅 거리·운행일 확인 | 10 |
| 경로 계산 | 목적지까지의 시간·거리와 시간 구성요소 | 25 |
| 결과 해석 | 평균, 사람별 차이, 차이가 생긴 이유 | 25 |
| 이동 재생 | 경로·이동시각·대기·도착 상태의 일치 | 25 |
| 장면 비교 | 지정 시각의 도착 인원과 선택한 사람의 상태 | 15 |

화면 디자인은 채점하지 않습니다.

추가 분석은 선택 사항이며 채점에 포함하지 않습니다.
한 사람의 환승 상한을 0회와 2회로 바꾸어 경로를 비교하거나, 7장의 요금 계산·수단 선택 모형을 적용해 볼 수 있습니다.

## 참고 실습

- [3장 다익스트라](https://github.com/jihoyeo/mobility-simulation-book/blob/main/labs/ch03_dijkstra.ipynb)
- [4장 속도와 통행시간](https://github.com/jihoyeo/mobility-simulation-book/blob/main/labs/ch04_speeds_engine.ipynb)
- [5장 GTFS](https://github.com/jihoyeo/mobility-simulation-book/blob/main/labs/ch05_gtfs.ipynb)
- [6장 RAPTOR](https://github.com/jihoyeo/mobility-simulation-book/blob/main/labs/ch06_raptor.ipynb)
- [7장 환승·요금과 수단 선택](https://github.com/jihoyeo/mobility-simulation-book/blob/main/labs/ch07_raptor_fare.ipynb)
